# Tarea 1 – Parte 2: Modelado y Clasificación

**Medio:** El Comercio  
**Número de noticias:** 360

## Objetivo

Comparar diferentes estrategias de clasificación automática de titulares de noticias
en cuatro categorías:

- politica
- economia
- deportes
- ciencia_tecnologia

Se evaluarán tres enfoques:

1. Clasificación zero-shot.
2. Etiquetado mediante un modelo de lenguaje (LLM).
3. Fine-tuning de un modelo Transformer ligero.

La evaluación final utilizará Accuracy, F1-macro y matriz de confusión.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [3]:
DATOS = Path("datos")

ruta_noticias = DATOS / "noticias.csv"

df = pd.read_csv(ruta_noticias)

print("Dimensión:", df.shape)

df.head()

Dimensión: (360, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Presidente del TC sobre leyes aprobadas por el...,El presidente del Tribunal Constitucional (TC)...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/presid...,2026-09-07T20:51:41-05:00,politica
1,Caso Árbitros-Odebrecht: audiencia de control ...,La Corte Superior Nacional de Justicia Penal E...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/caso-a...,2026-10-12T12:48:23-05:00,politica
2,Flavio Figallo del Partido del Buen Gobierno: ...,"El senador del Partido del Buen Gobierno, Flav...",El Comercio,Congreso,https://elcomercio.pe/politica/congreso/flavio...,2026-10-16T11:47:20-05:00,politica
3,Unión Europea y Minem acuerdan creación de mes...,La Unión Europea (UE) y el Ministerio de Energ...,El Comercio,Gobierno,https://elcomercio.pe/politica/gobierno/union-...,2026-10-13T19:19:04-05:00,politica
4,Fiscalía realizó diligencias en Ministerio de ...,Las diligencias estuvieron a cargo de la fisca...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/albert...,2026-08-27T13:43:00-05:00,politica


In [4]:
print(df["etiqueta"].value_counts())

print()
print("Valores nulos:", df.isna().sum().sum())
print("URLs duplicadas:", df["url"].duplicated().sum())

etiqueta
politica              95
economia              95
ciencia_tecnologia    95
deportes              75
Name: count, dtype: int64

Valores nulos: 0
URLs duplicadas: 0


In [5]:
muestra_zero_shot = (
    df.groupby("etiqueta", group_keys=False)
      .sample(n=10, random_state=SEED)
      .reset_index(drop=True)
)

print("Tamaño muestra zero-shot:", len(muestra_zero_shot))
print()
print(muestra_zero_shot["etiqueta"].value_counts())

Tamaño muestra zero-shot: 40

etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64


## 1. Baseline Zero-Shot

Se evaluará un modelo zero-shot sobre una muestra balanceada de 40 titulares,
con 10 ejemplos por cada categoría.

Las categorías candidatas son:
- politica
- economia
- deportes
- ciencia_tecnologia

El desempeño se medirá mediante Accuracy y F1-macro.

In [6]:
from transformers import pipeline

c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
clasificador_zero_shot = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\usuario\.cache\huggingface\hub\models--MoritzLaurer--mDeBERTa-v3-base-mnli-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\usuario\AppData\Local\Programs\Python\Python313\Lib\site-pack

In [8]:
etiquetas_modelo = {
    "politica": "política",
    "economia": "economía y negocios",
    "deportes": "deportes",
    "ciencia_tecnologia": "ciencia y tecnología"
}

candidate_labels = list(etiquetas_modelo.values())

candidate_labels

['política', 'economía y negocios', 'deportes', 'ciencia y tecnología']

In [9]:
predicciones_zero_shot = []
confianzas_zero_shot = []

for i, titular in enumerate(muestra_zero_shot["titular"]):

    resultado = clasificador_zero_shot(
        titular,
        candidate_labels=candidate_labels,
        hypothesis_template="Este titular pertenece a la categoría {}."
    )

    prediccion_texto = resultado["labels"][0]
    confianza = resultado["scores"][0]

    # Convertir nuevamente a nuestras etiquetas oficiales
    etiqueta_predicha = next(
        clave
        for clave, valor in etiquetas_modelo.items()
        if valor == prediccion_texto
    )

    predicciones_zero_shot.append(etiqueta_predicha)
    confianzas_zero_shot.append(confianza)

    print(
        f"{i + 1}/40 -> {etiqueta_predicha} "
        f"({confianza:.3f})"
    )

1/40 -> ciencia_tecnologia (0.386)
2/40 -> politica (0.307)
3/40 -> politica (0.506)
4/40 -> deportes (0.318)
5/40 -> economia (0.370)
6/40 -> ciencia_tecnologia (0.626)
7/40 -> ciencia_tecnologia (0.461)
8/40 -> politica (0.374)
9/40 -> politica (0.993)
10/40 -> economia (0.729)
11/40 -> deportes (0.982)
12/40 -> politica (0.635)
13/40 -> deportes (0.994)
14/40 -> deportes (0.997)
15/40 -> deportes (0.955)
16/40 -> deportes (0.528)
17/40 -> deportes (0.981)
18/40 -> deportes (0.497)
19/40 -> deportes (0.974)
20/40 -> deportes (0.998)
21/40 -> economia (0.856)
22/40 -> politica (0.587)
23/40 -> economia (0.991)
24/40 -> ciencia_tecnologia (0.627)
25/40 -> economia (0.978)
26/40 -> politica (0.755)
27/40 -> economia (0.827)
28/40 -> politica (0.936)
29/40 -> politica (0.679)
30/40 -> economia (0.974)
31/40 -> politica (0.913)
32/40 -> politica (0.940)
33/40 -> politica (0.991)
34/40 -> politica (0.503)
35/40 -> politica (0.998)
36/40 -> politica (0.960)
37/40 -> politica (0.908)
38/40 -

In [10]:
muestra_zero_shot["pred_zero_shot"] = predicciones_zero_shot
muestra_zero_shot["confianza_zero_shot"] = confianzas_zero_shot

In [11]:
accuracy_zero_shot = accuracy_score(
    muestra_zero_shot["etiqueta"],
    muestra_zero_shot["pred_zero_shot"]
)

f1_zero_shot = f1_score(
    muestra_zero_shot["etiqueta"],
    muestra_zero_shot["pred_zero_shot"],
    average="macro"
)

print("Accuracy Zero-Shot:", round(accuracy_zero_shot, 4))
print("F1-macro Zero-Shot:", round(f1_zero_shot, 4))

Accuracy Zero-Shot: 0.65
F1-macro Zero-Shot: 0.6328


In [12]:
muestra_zero_shot[
    [
        "titular",
        "etiqueta",
        "pred_zero_shot",
        "confianza_zero_shot"
    ]
].head(15)

,titular,etiqueta,pred_zero_shot,confianza_zero_shot
0,Tesla presenta un Roadster capaz de ‘volar’ y ...,ciencia_tecnologia,ciencia_tecnologia,0.386383
1,"Efecto cine, batería de emergencia o verificac...",ciencia_tecnologia,politica,0.306519
2,Millones de personas podrían tener que abandon...,ciencia_tecnologia,politica,0.506186
3,La nostalgia vuelve a encenderse: Atari relanz...,ciencia_tecnologia,deportes,0.318437
4,"Desarrollan un procedimiento “sencillo, sosten...",ciencia_tecnologia,economia,0.370386
5,"La IA preocupa al mundo, Europa le dice no a l...",ciencia_tecnologia,ciencia_tecnologia,0.625679
6,"Fallece Margaret Hamilton, pionera del program...",ciencia_tecnologia,ciencia_tecnologia,0.461208
7,"Google lanza Gemini 4 Argon, su nuevo modelo d...",ciencia_tecnologia,politica,0.374480
8,Trump quiere llamar “superinteligencia” a la I...,ciencia_tecnologia,politica,0.993262
9,¿Qué es Smart Connect y por qué puede ser una ...,ciencia_tecnologia,economia,0.729464


## 2. Etiquetado con LLM

Se utilizará un modelo de lenguaje para asignar una etiqueta a cada titular de la base completa.

Las únicas etiquetas permitidas son:

- politica
- economia
- deportes
- ciencia_tecnologia

Además de la etiqueta, se registrará una confianza estimada por el modelo.

In [13]:
df_llm = df[
    ["titular", "fecha", "etiqueta"]
].copy()

print("Noticias a etiquetar:", len(df_llm))
df_llm.head()

Noticias a etiquetar: 360


,titular,fecha,etiqueta
0,Presidente del TC sobre leyes aprobadas por el...,2026-09-07T20:51:41-05:00,politica
1,Caso Árbitros-Odebrecht: audiencia de control ...,2026-10-12T12:48:23-05:00,politica
2,Flavio Figallo del Partido del Buen Gobierno: ...,2026-10-16T11:47:20-05:00,politica
3,Unión Europea y Minem acuerdan creación de mes...,2026-10-13T19:19:04-05:00,politica
4,Fiscalía realizó diligencias en Ministerio de ...,2026-08-27T13:43:00-05:00,politica


In [14]:
ETIQUETAS_VALIDAS = {
    "politica",
    "economia",
    "deportes",
    "ciencia_tecnologia"
}

def limpiar_etiqueta(texto):
    texto = str(texto).strip().lower()

    reemplazos = {
        "política": "politica",
        "economía": "economia",
        "deporte": "deportes",
        "tecnologia": "ciencia_tecnologia",
        "tecnología": "ciencia_tecnologia",
        "ciencia y tecnologia": "ciencia_tecnologia",
        "ciencia y tecnología": "ciencia_tecnologia"
    }

    texto = reemplazos.get(texto, texto)

    if texto in ETIQUETAS_VALIDAS:
        return texto

    return None